# Chapter 1 &mdash; Three Machines, One Idea: Restrictions of the Turing Machine

**Concept 15 of the Chapter 1 decomposition:** *FA, PDA and LBA as Simplified Turing Machines*

Finite automata, pushdown automata and linear bounded automata are not separate inventions &mdash; they are <b>restrictions</b> of the Turing machine, one per pattern class.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter1/Concept-FA-PDA-LBA-As-Simplified-TMs/Concept-FA-PDA-LBA-As-Simplified-TMs.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_PDA        import *
from jove.Def_TM         import *
from jove.AnimateDFA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateDFA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateDFA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


| Machine | Memory | Recognises |
|---|---|---|
| **FA** (Rabin &amp; Scott, 1957) | none beyond the state | regular |
| **PDA** (Ginsburg, Greibach, 1960s) | one unbounded **stack** | context-free |
| **LBA** (Kuroda, 1960s) | tape **snipped** to the input | context-sensitive |
| **TM** (Turing, 1936) | unbounded tape | recursively enumerable |

Note the chronology: **the most general machine came first**, and the restrictions
arrived twenty years later, driven by practical parsing needs.

## 2. Definitions

### The same language, three ways: $a^n b^n$

A DFA cannot do it. A PDA can. A TM certainly can. Let us watch the DFA fail.

In [ ]:
# A DFA that tries to check a^n b^n -- and can only manage n <= 2
# Again mind the naming rule: the ACCEPTING state must start with 'F',
# and 'IF' means initial AND final (so the empty string is accepted).
tryab = md2mc('''DFA
IF : a -> A1
IF : b -> BH
A1 : a -> A2
A1 : b -> Fdone
A2 : a -> BH
A2 : b -> B1
B1 : b -> Fdone
B1 : a -> BH
Fdone : a | b -> BH
BH : a | b -> BH
''')
print("bounded a^n b^n DFA, states :", sorted(tryab["Q"]))
print("final states :", sorted(tryab["F"]), " <- must be non-empty, or it accepts nothing!")
assert tryab["F"], "a DFA with no F-named state accepts NOTHING -- a silent bug"

### The PDA: push each `a`, pop one per `b`

In [ ]:
anbn = md2mc('''PDA
IF : a , #  ; a#  -> Pa
Pa : a , a  ; aa  -> Pa
Pa : b , a  ; ''  -> Pb
Pb : b , a  ; ''  -> Pb
Pb : '' , # ; #   -> F
''')
print("a^n b^n PDA states :", sorted(anbn["Q"]))

<!-- nav-strip -->

---

&larr;&nbsp;[Ch1&nbsp;14.&nbsp;Pattern Class IV -- Recursively Enumerable Patterns](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter1/Concept-Recursively-Enumerable-Patterns/Concept-Recursively-Enumerable-Patterns.ipynb) &nbsp;&middot;&nbsp; [**Chapter 1** index](https://github.com/ganeshutah/Jove/blob/master/Chapter1/README.md) &nbsp;&middot;&nbsp; [Ch1&nbsp;16.&nbsp;Machine Classes as Programming Restrictions](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter1/Concept-Machines-As-Programming-Restrictions/Concept-Machines-As-Programming-Restrictions.ipynb)&nbsp;&rarr;

---

## 3. Tests

The DFA works up to its built-in bound, then breaks.

In [ ]:
for n in range(0, 5):
    s = 'a'*n + 'b'*n
    print("n=%d  %-10s DFA says %s" % (n, s, accepts_dfa(tryab, s)))
assert accepts_dfa(tryab, "aabb") and not accepts_dfa(tryab, "aaabbb")
print()
print("n=3 fails -- the DFA ran out of states, not out of legality.")
assert accepts_dfa(tryab, "aabb") and not accepts_dfa(tryab, "aaabbb")

The PDA has no such bound: the stack grows with the input.

In [ ]:
explore_pda("aaabbb", anbn, STKMAX=8)

The restriction ladder, as programming restrictions on C.

In [ ]:
print("FA  : finitely many FINITE variables, no heap, no recursion")
print("PDA : + functions that may recurse   (the call stack IS the stack)")
print("LBA : + unbounded memory, but only as much tape as the input")
print("TM  : + unbounded memory, freely accessed")
print()
print("CAUTION: TWO stacks, or ONE queue, already give full TM power.")
print("The 'single stack' restriction is what keeps a PDA a PDA.")

## 4. Animation


The finite automaton: no memory beyond its current state. Watch it &mdash; there is
nowhere for a count to live.

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(tryab, FuseEdges=True)

## 5. Exercises


1. Extend `tryab` to handle $n \le 4$. How many states per extra level?
2. The sidenote says two stacks give TM power. Sketch how two stacks simulate a tape.
   (Chapter 13 does this properly.)
3. Which machine would you need for "the same number of `a`s, `b`s **and** `c`s"?
   Try a PDA and see where the single stack runs out.

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for all 246 concepts.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter1/Concept-FA-PDA-LBA-As-Simplified-TMs')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')